In [ ]:
# Import important libraries
import numpy as np
import pandas as pd

# This option would allow me to see the long review text without getting cut off
pd.set_option('display.max_columns', None, 'display.max_colwidth', None)

# Pendahuluan
<p>Dalam notebook ini, saya melakukan task Analisis sentimen terhadap ulasan produk online yang dibuat oleh customer yang berbelanja pada platform Tokoepedia. Analisis sentimen ini bertujuan untuk menggali informasi lebih dalam yang terdapat dalam kumpulan ulasan, informasi ini dapat digunakan sebagai pengetahuan yang dapat dimanfaatkan oleh penjual maupun pihak-pihak yang terlibat dalam pengembangan platform belanja online. Dari ulasan produk yang telah di-analisa nanti, penjual dan pengembang platform online diharapkan dapat membuat informed decision dalam mengembangkan produk mereka, dan juga mengembangkan pelayanan mereka terhadap customer.</p>

<p>Notebook ini dibuat untuk keperluan menulis paper penelitian yang menjadi tugas saya di mata kuliah Pengenalan Pola. Dengan harapan saya dapat menggunakan hasil notebook ini untuk menulis paper penelitian dan mem-publish paper tersebut. Selain itu, saya juga berharap untuk memperluas ilmu yang saya miliki dari melakukan penelitian ini.</p>

<p>Sebagai mahasiswa informatika, saya yakin notebook yang saya buat masih jauh dari kata baik. Saya mohon maaf bila informasi yang disampaikan disini terkesan berantakan dan tidak beraturan, tapi saya akan berusaha untuk merapihkannya sedikit-demi sedikit di versi yang akan datang. Saya harap notebook ini dapat bermanfaat bagi yang membacanya.</p>

# 1. Business Understanding

<p>Dengan pesatnya perkembangan teknologi internet saat ini, banyak hal mulai mengalami transisi ke proses digital dan online. Salah satunya adalah kegiatan berbelanja. Platform belanja online telah berkembang dari tahun 2010-an hingga sekarang dan semakin banyak digunakan oleh masyarakat luas sebagai salah satu pilihan utama untuk berbelanja kebutuhannya. Hal ini disebabkan oleh kemudahan yang disediakan oleh platform belanja online yang memungkinkan pengguna berbelanja dimanapun dan kapanpun dengan menggunakan perangkat yang mereka miliki, seperti komputer maupun handphone. Selain itu, harganya yang seringkali lebih murah, dan juga berbagai promo yang disediakan oleh penjual maupun platform belanja online makin menguatkan minat masyarakat untuk berbelanja secara online.</p>

<p>Namun, seiring dengan perkembangannya yang pesat, bukan berarti kegiatan belanja online ini tanpa kekurangan. Ada banyak masalah yang seringkali muncul dalam proses belanja online ini yang dapat menurunkan kepuasan serta kepercayaan pengguna dalam melakukan kegiatan belanja online. Beberapa contoh masalah tersebut adalah produk yang tidak sesuai dengan deksripsi yang disediakan oleh penjual, pengiriman yang lambat dan bahkan bermasalah, pelayanan penjual yang buruk terhadap customer, dan bahkan terkadang terjadi penipuan yang dilakukan oleh seller. Hal ini tentu perlu diselesaikan untuk meningkatkan kepuasan pengguna yang nantinya juga akan meningkatkan frequensi belanja mereka.</p>

# 2. Data Understanding
<p>Dataset yang kami gunakan adalah dataset yang berisi data ulasan produk online yang dibuat oleh customer ketika selesai membeli suatu produk di platform Tokopeida. Data ini kami ambil dari hasil penelitian Rhio Sutoyo, et al. yang berjudul: <a href="https://www.sciencedirect.com/science/article/pii/S2352340922007612">"PREDECT-ID: Indonesian Product Reviews Dataset for Emotions Classification Tasks"</a></p>

In [ ]:
dataset = "/kaggle/input/predect-id/PRDECT-ID Dataset.csv"
df = pd.read_csv(dataset)

# Tampilkan 5 data ulasan random
df.sample(n=5, random_state=48)

In [ ]:
# Tampilkan informasi terkait dataset
df.info()

# Hitung jumlah kategori sentimen
df.Sentiment.value_counts()

<p>Melihat dari hasil di atas, terdapat 11 kolom/fitur yang terdapat dalam dataset ulasan yang saya gunakan. Karena task yang yang lakukan adalah analisis sentimen yang merupakan salah satu task dari kategori NLP, maka fitur yang saya gunakan untuk melakukan klasifikasi sentimen hanya "Customer Review" dan "Sentiment". Sisanya dapat digunakan untuk melakukan analisis lebih lanjut. </p>

<p>Jumlah ulasan berjumlah 5400 data, dengan ulasan yang memiliki sentimen negatif berjumlah 2821 ulasan dan ulasan positif berjumlah 2579 ulasan. Perbandingan ulasan bersentimen positif dan negatif cukup seimbang, sehingga untuk saat ini tidak perlu dilakukan balancing terhadap dataset.</p>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(y="Category", hue="Sentiment", palette="deep", data=df)
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

<p>Melihat dari graph diatas, rata-rata kategori memiliki perbandingan class 50:50 seperti "Books" dan "Kitchen", beberapa kategori juga ada yang memiliki perbandingan 80:120 seperti "Computers and Laptops", "Toys and Hobbies", dan "Food and Drink". Hal ini sangat berguna untuk membangun model klasifikasi sentimen yang dapat membedakan antara ulasan negatif dan positif, dan juga luasnya kategori yang ada membuat model yang akan dibangun nanti dapat memahami kosakata yang luas.</p>

<p>Walaupun sepertinya beberapa kategori hanya memiliki ulasan positif seperti "Precious Metal" dan "Tour and Travel". Hal ini bisa jadi disebabkan karena tidak banyaknya produk dengan kategori tersebut yang dijual, dan ulasan negatif hampir tidak ada pada saat peneliti yang membuat dataset menghimpun dataset ulasannya.</p>

# 3. Data Preparation

<p>Dalam tahap ini, saya akan mempersiapkan data agar siap untuk digunakan sebagai data training model klasifikasi sentimen yang akan dibuat nanti. Tahapan ini terdiri dari: case folding to lowercase, word normalization, stopwords removal, symbol and punctuation removal, dan terakhir feature extraction dengan TF-IDF.</p>

<h3>a. Case folding</h3>
<p>Case folding adalah proses dimana karakter alfabet diseragamkan 'case'nya menjadi satu 'case' yang sama. Di notebook ini saya mengubah case dari huruf di setiap ulasan ke dalam bentuk huruf kecil agar kata yang sama tidak menjadi fitur baru hanya karena perbedaan bentuk huruf.</p>

In [ ]:
# Buat dataframe baru untuk menampung data yang sudah dilakukan preprocessing
clean_df = df.copy()

clean_df['Customer Review'] = clean_df['Customer Review'].str.lower()

In [ ]:
print(f"Before: '{df['Customer Review'][4]}'")
print(f"After: '{clean_df['Customer Review'][4]}'")

<h3>b. Word normalization</h3>
<p>Dalam tahap ini saya melakukan normalisasi bentuk kata, untuk mengurangi redundansi fitur kata yang terbentuk. Normalisasi disini saya lakukan dengan cara mengubah kata-kata yang berupa singkatan menjadi kata formal dengan makna yang sama. Contoh: ngga -> tidak, yg -> yang, dan seterusnya.</p>

In [ ]:
# Import regular expression library
import re

word_mapping = {
    "yg": "yang","tp": "tapi","bgt": "begitu","jg": "juga","dgn": "dengan","pake": "pakai","jd": "jadi","klo": "kalo",
    "lg": "lagi","dr": "dari","utk": "untuk","gk": "tidak","sdh": "sudah","ngga": "tidak","brg": "barang","ga": "tidak",
    "gak": "tidak","rapi": "rapih","cpt": "cepat","krn": "karena","sy": "saya","tdk": "tidak","nggak": "tidak","kalo": "kalau",
    "cepet": "cepat","pake": "pakai","gitu": "begitu","udh": "udah","d": "di","g" : "tidak","tgl": "tanggal","pake": "pakai",
    "sampe": "sampai","mantab": "mantap"
    # Add more mappings as needed
}

# Define a function to replace shortened words using re.sub
def normalize_word(text):
    # Create a regex pattern from the keys of the word_mapping dictionary
    pattern = r'\b(' + '|'.join(re.escape(word) for word in word_mapping.keys()) + r')\b'

    # Define a function to replace matches with their corresponding values from the dictionary
    def replace(match):
        return word_mapping[match.group(0)]

    # Use re.sub with the pattern and the replace function
    normalized_text = re.sub(pattern, replace, text)

    return normalized_text

clean_df['Customer Review'] = clean_df['Customer Review'].apply(normalize_word)

<h3>c. Stopword Removal</h3>
<p>Stopword merupakan kata yang sangat sering digunakan dalam sebuah kalimat, namun biasanya tidak memiliki makna yang berarti, contohnya seperti: yang, adalah, akan, dan sebagainya. Stopword list yang saya gunakan disini diambil dari library NLTK. Namun perlu diingat, tidak semua stopword yang terdapat dalam list library tersebut perlu dihilangkan. Ada beberapa kata yang bisa dibilang cukup penting dan dapat mengandung nilai sentimen. Contohnya seperti 'baik', 'sangat', 'lama', dan sebagainya.</p>

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')

# Create an Indonesian stopword list
stopwords_list = stopwords.words('indonesian')

<h4>c.1 Adjusting the stopwords list</h4>
<p>Untuk mencegah kata yang memiliki nilai sentimen ikut terdeteksi sebagai stopwords, maka perlu dilakukan perubahan/adjustment terhadap stopword list default yang diambil dari nltk. Selain kata yang memiliki nilai sentimen, ada beberapa kata yang bisa dibilang tidak penting yang saya temukan dalam dataset, sehingga saya menambahkan kata-kata tersebut ke dalam stopword list.</p>

In [ ]:
# Search for important words in the stopwords list
search_key = "baik"
try:
    key_index = stopwords_list.index(search_key)
    print(f"Kata yang dicari : '{search_key}' berada dalam index {key_index}")
except ValueError:
    print(f"Kata yang dicari : '{search_key}' tidak ditemukan")

In [ ]:
# Important words that is needed to keep
stopwords_to_keep = ['baik', 'biasa', 'bukan', 'amat', 'baru', 'biasa', 'bukan', 'cukup', 'kurang', 'lama', \
                           'sangat', 'sedikit', 'tak', 'tidak', 'kali']
new_stopwords_list = [word for word in stopwords_list if word.lower() not in stopwords_to_keep]

# Add more irrelevant words into the stopwords list
words_to_remove = ['gan', 'nya', 'aja', 'sih', 'deh', 'n', 'dah', 'ya', 'gitu', 'pa', 'kalo', 'udah', 'kali']
new_stopwords_list.extend(words_to_remove)

<h4>c.2 Remove stopwords from review text</h4>
Setelah stopword list selesai di-adjust. Saatnya menghilangkan kata-kata yang terindikasi sebagai stopwords.

In [ ]:
# Remove stopwords
stop_words = set(new_stopwords_list)

def remove_stopwords(text):
    tokens = word_tokenize(text)  # Tokenize the text
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]  # Remove stopwords
    cleaned_text = ' '.join(filtered_tokens)  # Join filtered tokens back into a cleaned text
    return cleaned_text.strip()  # Strip leading and trailing spaces

clean_df['Customer Review'] = clean_df['Customer Review'].apply(remove_stopwords)

<h3>d. Remove other symbols and punctuations</h3>

<p>Dalam teks ulasan, banyak terdapat tanda baca seperti '.', ',', '!', '?', dan sebagainya. Dalam beberapa kasus, penggunaan beberapa tanda baca seperti '!' dan '?' dapat mengubah makna ulasan yang ditulis ataupun meningkatkan intensitas sentimen yang ingin disampaikan.</p>

<p>Namun, karena hasil akurasi yang ada sudah cukup tinggi tanpa melakukan ini, maka untuk saat ini semua tanda baca tidak saya gunakan dalam proses klasifikasi sentimen.</p>

In [ ]:
def remove_special_characters(text):
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove all non-alphanumeric characters except whitespace
    return cleaned_text.strip()  # Strip leading and trailing spaces

clean_df['Customer Review'] = clean_df['Customer Review'].apply(remove_special_characters)

<h3>[Experimental]Word Stemming</h3>

In [ ]:
# Stem words
# !pip install PySastrawi

In [ ]:
# # import StemmerFactory class
# from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# # create stemmer
# factory = StemmerFactory()
# stemmer = factory.create_stemmer()

# def stem_words(text):
#     stemmed_text = stemmer.stem(text)  # Join filtered tokens back into a cleaned text
#     return stemmed_text  # Strip leading and trailing spaces

# clean_df['Customer Review'] = df['Customer Review'].apply(stem_words)

<h4>e. Encode Sentiment Labels</h4>
<p>Agar sentiment dapat diproses untuk training model, maka label sentimen perlu di encode ke dalam bentuk angka yang mana disini saya ubah label 'Positive' menjadi 1, dan label 'Negative' menjadi 0.</p>

In [ ]:
# Encode sentiment labels. 'Positive' to 1 and 'Negative' to 0
# clean_df['Sentiment'] = clean_df['Sentiment'].replace({'Positive':1, 'Negative': 0})

<h3>f. Analyzing the data</h3>
<p>Setelah melakukan preprocessing terhadap teks ulasan, selanjutnya saya mencoba untuk menganalisa karakteristik ulasan yang ada dengan menggunakan tools visualisasi data. </p>

<h4>f.1 Most frequent words</h4>

In [ ]:
# Make a copy of cleaned dataframe for visualization purpose
df_vis = clean_df.copy()

from collections import Counter
df_vis['temp_list'] = df_vis['Customer Review'].apply(lambda x:str(x).split())
top = Counter([item for sublist in df_vis['temp_list'] for item in sublist])
temp = pd.DataFrame(top.most_common(20))
temp.columns = ['Common_words','count']
temp.style.background_gradient(cmap='Blues')

<p>Melihat dari 20 kata teratas yang sering muncul, dalam ulasannya pengguna sering menyebut kata 'barang', 'sesuai', 'pengiriman', 'cepat', 'bagus', dan sebagainya. Hal ini menunujukkan bahwa pelanggan sangat memperhatikan kualitas barang yang diterima, serta proses pengiriman yang cepat juga mempengaruhi kepuasan pelanggan.

<h4>f.2 Wordcloud</h4>

In [ ]:
# Split the dataframe into positive and negative only
df_vis_pos = df_vis[df_vis['Sentiment'] == "Positive"]
df_vis_neg = df_vis[df_vis['Sentiment'] == "Negative"]

In [ ]:
from wordcloud import WordCloud

# Create WordCloud plot for positive reviews
txt = ' '.join(rev for rev in df_vis_pos['Customer Review'])
plt.figure(figsize=(15,8))

wordcloud = WordCloud(
            background_color = 'black',
            max_font_size = 100,
            max_words = 100,
            width = 800,
            height = 600
            ).generate(txt)


plt.imshow(wordcloud,interpolation = 'bilinear')
plt.axis('off')
plt.title(label="Wordcloud ulasan positif", fontsize=20)
plt.show()

In [ ]:
# Create WordCloud plot for negative reviews
txt = ' '.join(rev for rev in df_vis_neg['Customer Review'])
plt.figure(figsize=(15,8))

wordcloud = WordCloud(
            background_color = 'black',
            max_font_size = 100,
            max_words = 100,
            width = 800,
            height = 600
            ).generate(txt)


plt.imshow(wordcloud,interpolation = 'bilinear')
plt.axis('off')
plt.title(label="Wordcloud ulasan negatif", fontsize=20)
plt.show()

# 4. Modelling
<p>Dalam tahap ini saya mencoba untuk membangun model untuk melakukan klasifikasi sentimen.</p>

<h4>4.a Split Dataset</h4>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
# Split data ke train dan test
reviews = clean_df['Customer Review']

X = tfidf_vectorizer.fit_transform(reviews)
y = clean_df['Sentiment']


X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.3, random_state= 42)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

svm_clf = SVC(kernel='linear')
svm_clf.fit(X_train, y_train)

In [ ]:
y_pred = svm_clf.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# Classifier Optimalization
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10, 100],         # Regularization parameter
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],  # Kernel type
    'degree': [2, 3, 4],           # Degree for poly kernel
    'gamma': ['scale', 'auto'],    # Kernel coefficient
}

grid_search = GridSearchCV(SVC(), param_grid, cv=10, scoring='accuracy', verbose=0, n_jobs=-1)

grid_search.fit(X_train, y_train)

# Step 8: Get the best parameters and best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f"Best parameters: {best_params}")

y_pred = best_model.predict(X_test)

# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

In [ ]:
best_model.classes_

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_test, y_pred, labels=best_model.classes_)
cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
cm_disp.plot(cmap=plt.cm.Blues)
plt.show()

# Naive Bayes Version

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)

In [ ]:
# Cross validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(estimator=nb_clf, X=X_train, y=y_train, cv=10)
average_score = scores.sum()/10
print(average_score)

In [ ]:
nb_clf.classes_

In [ ]:
nb_pred = nb_clf.predict(X_test)

In [ ]:
# Confusion Matrix MultinomialNB
cm = confusion_matrix(y_test, nb_pred, labels=nb_clf.classes_)
cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nb_clf.classes_)
cm_disp.plot(cmap=plt.cm.Blues)
plt.show()

# Sampel untuk demonstrasi proses preprocessing

In [ ]:
data_sample = clean_df[['Customer Review', 'Sentiment']]

In [ ]:
data_sample['Review Length'] = data_sample['Customer Review'].apply(lambda x: len(str(x).split()))

In [ ]:
# data_sample[data_sample['Sentiment'] == 'Positive']['Customer Review'].sample(20)
# Data nomor 1415 cukup baik untuk jadi sampel
text_sample = data_sample['Customer Review'][1415]
print(text_sample)

In [ ]:
# Jadikan lowercase
text_sample_clean = text_sample.lower()
print(text_sample_clean)

In [ ]:
# Hilangkan special character
text_sample_clean = remove_special_characters(text_sample_clean)
print(text_sample_clean)

In [ ]:
# Hilangkan stopwords
text_sample_clean = remove_stopwords(text_sample_clean)
print(text_sample_clean)

In [ ]:
# Stemming
# text_sample_clean = stem_words(text_sample_clean)
# print(text_sample_clean)

# Future work and further improvement
1. Ada beberapa kata yang disingkat-singkat dan bukan proper formnya, jadi diusahakan kata tersebut diperbaiki untuk meng-improve hasil klasifikasi sentimen. Contoh : seler -> seller, yg -> yang.
2. Consider using word stemming, or maybe word lemmatization?
3. Ada beberapa kata yang memiliki bentuk yang berbeda tapi makna tetap sana (idk what the term for it lmao). Contoh : mantap -> mantab, mantabbb. Like, repeating alphabets? I'm sure I've read this kind of problem before on a journal. After reading some papers, I know that this process is called **normalization**
4. Ada beberapa kata berbahasa inggris seperti "thanks", "recomended", terkadang ada pula yang mencampurkan bahasa Indonesia dan bahasa Inggris. Apakah ini perlu ditangani?
5. Ada juga beberapa data review yang hanya memiliki 1 kata, dan bisa dibilang sangat pendek untuk sebuah review. Baris yang seperti ini perlu diapakan? Apakah dihilangkan atau di-keep?
6. Cek stopwords list apakah ada kata yang diperlukan namun terdapat dalam stopwords? -> **"tidak", "kurang"**
7. Consider checking SVM performance since there's a lot of paper using that algorithm -> **Hasilnya cenderung sama, akurasi hanya meningkat 1% di svm. Dan untuk metrics lainnya sama.**

<br>EDA :
1. Cek distribusi kelas negatif dan kelas positif dengan seaborn countplot. Cek juga apakah terjadi imbalance (tapi sepertinya tidak, karena classification report tidak menunjukkan anomali)
2. Cek jumlah kata di setiap review dan distribusinya. Apakah banyak review yang singkat atau banyak review yang panjang.
3. Kata apakah yang paling sering di mention dalam keseluruhan review?
4.

<br>**Further note about preprocessing**:
1. List of words need to be removed : yg, n, gan, jg, sih, tp, deh

Notes :
1. Datasets doesnt contain any emojis, but there are some symbols that maybe is a replacement for emojis that already cleaned by the dataset author

# Experimental Stuff

In [ ]:
# Trying regex to find something
def singkatan(text):
    word = text.split()
    for w in word:
        return ' '.join(word)
singkatan("yg saya hormati")

In [ ]:
# Dictionary to map shortened words to their proper forms
word_mapping = {
    "yg": "yang",
    "tp": "tapi",
    "bgt": "begitu"
    # Add more mappings as needed
}

# Sample text with shortened words
text = "Saya suka yg begini, tp yg bgt juga bagus."

# Define a function to replace shortened words using re.sub
def normalize_shortened_words(text):
    # Create a regex pattern from the keys of the word_mapping dictionary
    pattern = r'\b(' + '|'.join(re.escape(word) for word in word_mapping.keys()) + r')\b'

    # Define a function to replace matches with their corresponding values from the dictionary
    def replace(match):
        return word_mapping[match.group(0)]

    # Use re.sub with the pattern and the replace function
    normalized_text = re.sub(pattern, replace, text)

    return normalized_text

# Call the function to normalize the text
normalized_text = normalize_shortened_words(text)

print(normalized_text)

In [ ]:
pattern = r'\bkaya\b'
matching_rows = clean_df[clean_df['Customer Review'].str.contains(pattern, regex=True)]

print(matching_rows['Customer Review'].sample(10))